# Product-wise Word Cloud Generator for Amazon Reviews

In this project, you will build a **product-wise word cloud generator** using Amazon Reviews data. This generator will create word clouds based on the most frequent words in customer reviews for each unique product. Each word cloud will show up to the 50 most common words for a product, giving insights into how customers describe and perceive it.

### Project Objectives

By completing this project, you will:
- **Load and explore a dataset** using the `datasets` library.
- **Preprocess text data** by converting it to lowercase and removing punctuation.
- **Remove stop words** to keep only meaningful words.
- **Group reviews by product** using the `parent_asin` field.
- **Generate word clouds** with the most frequent words for each product.

You can read more about the dataset [here on Hugging Face](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023).

---

### Dataset Information

We will use the **Amazon Reviews 2023** dataset, which contains reviews from Amazon across various product categories. Each review includes fields such as rating, title, review text, images, product ID (`asin`), `parent_asin` (product group ID), and other details.

#### Relevant Fields for This Project
- **parent_asin**: A unique identifier for each product group.
- **text**: The body of the customer review.
- **title**: The title of the review.

Each review provides insights about a product. By analyzing these reviews, we can determine the common words associated with each product group.

---

### Step-by-Step Guide

Follow these steps to complete the project:

---

### Step 0: Install Required Libraries



In [1]:
!pip install datasets==3.1.0



### Step 1: Load the Amazon Reviews Dataset

First, load the Amazon Reviews dataset from Hugging Face using the `datasets` library. We’ll print out the first entry to get a sense of the data structure.

#### Task 1: Load and inspect the dataset.



In [2]:
import json
from datasets import load_dataset, Dataset

def load_amazon_reviews() -> Dataset:
    """
    Loads the Amazon Reviews dataset from Hugging Face.

    Returns:
        dataset (Dataset): The full dataset loaded from Hugging Face.
    """
    dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_All_Beauty", trust_remote_code=True)
    return dataset

# ================================================
# Uncomment the following lines for the first run
# ================================================

# Load the dataset and print a sample review
dataset = load_amazon_reviews()

# dataset.save_to_disk("dataset")

for i in range(3):
    print(json.dumps(dataset["full"][i], indent=2), "\n")

{
  "rating": 5.0,
  "title": "Such a lovely scent but not overpowering.",
  "text": "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!",
  "images": [],
  "asin": "B00YQ6X8EO",
  "parent_asin": "B00YQ6X8EO",
  "user_id": "AGKHLEW2SOWHNMFQIJGBECAF7INQ",
  "timestamp": 1588687728923,
  "helpful_vote": 0,
  "verified_purchase": true
} 

{
  "rating": 4.0,
  "title": "Works great but smells a little weird.",
  "text": "This product does what I need it to do, I just wish it was odorless or had a soft coconut smell. Having my head smell like an orange coffee is offputting. (granted, I did know the smell was described but I was hoping it would be light)",
  "images": [],
  "asin": "B081TJ8YS3",
  "parent_asin": "B081TJ8YS3",
  "user_id": "AGKHLE

In [3]:
# ================================================
# Uncomment the following lines after the first run
# ================================================
# from datasets import load_from_disk

# dataset = load_from_disk("dataset")
# for i in range(3):
#     print(json.dumps(dataset["full"][i], indent=2), "\n")


Step 2: Preprocess Text

To prepare the review text for analysis, convert it to lowercase and remove any punctuation. This standardizes the text data, allowing for better frequency analysis.

Task 2: Implement the preprocess_text function to clean the text.



In [4]:
import string

def preprocess_text(text) -> str:
    """
    Preprocesses the text by converting it to lowercase and removing punctuation.

    Args:
        text (str): The raw text input from a review.

    Returns:
        str: The cleaned and preprocessed text.
    """
    # Hint: You can use the `str.translate()` method to remove punctuation and string.punctuation for the list of punctuation characters.

    # convert to lowercase
    text= text.lower()

    # Instantiate a translator object that removes punctuation from a string
    translator = str.maketrans("","",string.punctuation)

    # Apply the translator to the text and return the result
    return text.translate(translator)

# Example usage
sample_text = dataset["full"][0]["text"]
processed_text = preprocess_text(sample_text)
print("Processed text:", processed_text)

Processed text: this spray is really nice it smells really good goes on really fine and does the trick i will say it feels like you need a lot of it though to get the texture i want i have a lot of hair medium thickness i am comparing to other brands with yucky chemicals so im gonna stick with this try it



Step 3: Remove Stop Words

Stop words are common words (like “the”, “is”, and “and”) that usually don’t carry significant meaning. Removing them helps focus on more meaningful words.

Task 3: Implement the remove_stop_words function to filter out stop words.



In [5]:
!pip install nltk

In [6]:
import nltk
from nltk.corpus import stopwords

# Download the stopwords dataset if not already downloaded
nltk.download("stopwords")

# Load English stop words
stop_words = set(stopwords.words("english"))

def remove_stop_words(text) -> list:
    """
    Removes common stop words from the preprocessed text.

    Args:
        text (str): The cleaned text input from a review.

    Returns:
        list: A list of words with stop words removed.
    """
    words_list = text.split()

    return [word for word in words_list if word not in stop_words]

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
# Test result
result = remove_stop_words(processed_text)
print(f"The first 5 words from stopwords: {list(stop_words)[:5]}")
print(f"The first 5 words from the list of words without stopwords: {result[:5]}")
print(f"Stop words in the result: {len([word for word in result if word  in stop_words])}")

The first 5 words from stopwords: ["you'd", 'isn', "mustn't", 'wasn', 'very']
The first 5 words from the list of words without stopwords: ['spray', 'really', 'nice', 'smells', 'really']
Stop words in the result: 0



Step 4: Group Reviews by parent_asin

Group all review texts by product (parent_asin), so that each product has a combined text of all its reviews. This allows us to analyze each product’s word frequency in a comprehensive way.

Task 4: Implement the group_reviews_by_product function to organize reviews by product.



In [8]:
from collections import defaultdict
from tqdm import tqdm

def group_reviews_by_product(dataset) -> dict:
    """
    Groups reviews by `parent_asin`, concatenating review text for each product.

    Args:
        dataset (Dataset): The dataset containing reviews.

    Returns:
        dict: A dictionary with `parent_asin` as keys and concatenated review text as values.
    """
    # Hint: You can use a defaultdict to group reviews by parent_asin.
    result = defaultdict(list)

    for row in tqdm(dataset['full']):
        result[row["parent_asin"]].append(row["text"])

    return {parent_asin: " ".join(text_list) for parent_asin, text_list in result.items()}

In [9]:
# ===========
# test result
# ===========

# Example usage
product_reviews = group_reviews_by_product(dataset)
print("Sample concatenated review text for a product:", list(product_reviews.values())[0])

100%|██████████| 701528/701528 [01:22<00:00, 8551.09it/s]


Sample concatenated review text for a product: This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it! This does not work as well as other sea mist hair products. Does not give the beach hair look at all. Makes my hair smell amazing and I love the texture it adds Pros: smells AMAZING like a beachy beverage! (I’d say similar to a margarita), defines my wavy hair really well- especially on day two if I lightly wet and respray my hair then re-plop it, all natural ingredients that are Earth friendly, glass bottle that’s infinitely recyclable (unlike plastic), my curls are soft rather than crunchy.<br /><br />Cons: the yummy smell isn’t super strong after it dries, frizz control isn’t perfect<br /><br />All in all I’m happy with it and will buy it ag


Step 5: Generate Word Cloud (Top 50 Words) for Each Product

Now, we’ll count the word frequencies for each product, and keep only the top 50 words by frequency.

Task 5: Implement the generate_word_cloud function to return the top 50 words by frequency.



In [10]:
# Define global parameter for top K words
TOP_K = 50

def generate_word_cloud(words, top_k=TOP_K) -> dict:
    """
    Generates a word cloud dictionary with the top K most frequent words.

    Args:
        words (list): List of words to count.
        top_k (int): Number of most frequent words to return.

    Returns:
        dict: Dictionary of top K word frequencies.
    """
    word_count = {}

    # Create a dictionary where keys are words and values are their counts
    for word in words:
        if word not in word_count:
            word_count[word] = 0
        word_count[word] += 1

    # Sort the dictionary by its values using `sorted` method
    # then slice it using "top_k" parameter
    sorted_dict = dict(
        sorted(word_count.items(), key=lambda item: item[1], reverse=True)[:top_k]
        )

    return sorted_dict

In [11]:
# ===========
# Test result
# ===========

# apply the function to the preprocessed text of the first review
generate_word_cloud(result, 10)

{'really': 3,
 'lot': 2,
 'spray': 1,
 'nice': 1,
 'smells': 1,
 'good': 1,
 'goes': 1,
 'fine': 1,
 'trick': 1,
 'say': 1}


Step 6: Combine Everything to Generate Word Clouds by Product

Finally, combine the functions to generate word clouds for each product in the dataset.

Task 6: Use generate_product_word_clouds to create word clouds by product.



In [12]:
def generate_product_word_clouds(dataset, top_k=TOP_K) -> dict:
    """
    Generates a word cloud for each unique product in the dataset.

    Args:
        dataset (Dataset): The Amazon reviews dataset.
        top_k (int): Number of top words to include in each product's word cloud.

    Returns:
        dict: Dictionary where each key is `parent_asin` and the value is a dictionary of top K word frequencies.
    """
    # First get a dictionary whose keys are `parent_asin` and values are concatenated reviews
    grouped = group_reviews_by_product(dataset)

    # Second preprocess and remove the values of the `grouped` which are concatenated reviews
    preprocessed = {parent_asin: remove_stop_words(preprocess_text(reviews)) for parent_asin, reviews in grouped.items()}

    # Finally create word cloud from words list of each product and return the resulting dictionary
    return {parent_asin: generate_word_cloud(words_list, top_k) for parent_asin, words_list in preprocessed.items()}

In [14]:
# Generate the word clouds and print a sample result
product_word_clouds = generate_product_word_clouds(dataset)
sample_asin = list(product_word_clouds.keys())[0]
print(f"Top words for product {sample_asin}:")
print(product_word_clouds[sample_asin])

100%|██████████| 701528/701528 [01:25<00:00, 8198.86it/s] 


Top words for product B081TJ8YS3:
{'hair': 14, 'love': 10, 'product': 9, 'dry': 9, 'smell': 7, 'like': 7, 'shampoo': 6, 'powder': 6, 'great': 6, 'dark': 5, 'used': 5, 'would': 4, 'one': 4, 'i’ve': 4, 'tried': 4, 'days': 4, 'scent': 4, 'better': 4, 'orange': 3, 'coffee': 3, 'shade': 3, 'br': 3, 'said': 3, 'works': 3, 'well': 3, 'shampoos': 3, 'scalp': 3, 'soft': 2, 'head': 2, 'came': 2, 'red': 2, 'could': 2, 'see': 2, 'trying': 2, 'highly': 2, 'problem': 2, 'immediately': 2, 'natural': 2, 'look': 2, 'really': 2, 'wanted': 2, 'heavy': 2, 'also': 2, 'day': 2, 'didnt': 2, 'stuff': 2, 'every': 2, 'blonde': 2, 'lets': 2, 'without': 2}


---
### Summary

In this project, you completed a product-wise word cloud generator by:

1.	Loading the Amazon Reviews dataset.
2.	Preprocessing and cleaning the text data.
3.	Removing stop words.
4.	Grouping reviews by parent_asin.
5.	Generating word clouds with the top 50 most frequent words for each product.

This provides insights into customer feedback on each product, showcasing the words most associated with each item.

### How to Submit the Project

1. Create a Jupyter Notebook file on your Google Drive.
2. Write your solution in the notebook.
3. Share access with your mentor (or provide a link with editing rights).
4. Submit the link to the notebook file or the folder as your solution for this lesson.